# Extraction

1. **Extract** structured JSON (`job_title`, `experience`, `education`,
   `other_info`) from each candidate's raw job description text, using the
   local Qwen few-shot pipeline.


In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.model_selection import train_test_split

In [2]:
# ---- Config ----
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
CLEANED_DATA_PATH = "/kaggle/input/datasets/divyeshbhavsar/candidates-cleaned/candidates_cleaned.csv"   # output of the wrangling notebook
EXTRACTED_OUTPUT_PATH = "/kaggle/working/candidate_extracted.json"

SPLIT_OUTPUT_DIR = "/content/drive/MyDrive/Colab Data"
MIN_ROWS_PER_TITLE = 15    # titles with fewer rows than this get merged into one bucket for stratification
TEST_SIZE = 0.15
VAL_SIZE_OF_REMAINDER = 0.176   # ~0.15 of the original total once test is already removed
SPLIT_SEED = 42

MAX_NEW_TOKENS_EXTRACT = 90
DO_SAMPLE = False
LOAD_IN_4BIT = False

# Batched generation -- with ~1200 rows, one-row-at-a-time generation means
# 1200 separate forward passes; batching lets the GPU process several
# candidates in a single model.generate() call instead. Start at 8 and
# raise it if GPU memory allows -- larger batches process more rows per
# call, but each call also costs more memory (roughly proportional to
# batch_size x sequence_length).
EXTRACT_BATCH_SIZE = 25

In [3]:
# ---- Load tokenizer + model ----
quantization_kwargs = {}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    quantization_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Batched generation on a decoder-only model requires LEFT padding: every
# sequence in a batch must end at the same position so generation can
# continue from one shared point for the whole batch. Right-padding (the
# tokenizer default) would leave padding *after* each real sequence,
# which breaks that assumption.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype="auto", **quantization_kwargs
)
print(f"Loaded {MODEL_NAME} on {next(model.parameters()).device}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-3B-Instruct on cuda:0


In [4]:
# ---- Load the cleaned dataset ----
df = pd.read_csv(CLEANED_DATA_PATH)
print(df.shape)
df.head()

(1266, 4)


,id,title,location,screening_score
0,1,innovative and driven professional seeking a r...,United States,100
1,2,ms applied data science student usc research a...,United States,100
2,3,computer science student seeking full-time sof...,United States,100
3,4,microsoft certified power bi data analyst mba ...,United States,100
4,5,graduate research assistant at uab masters in ...,United States,100


## Step 1: Extract structured JSON from each raw job description

Same schema and few-shot prompt as the extraction stage built earlier:
`job_title` (abbreviations normalized), `experience` (only if stated or
implied in the text itself), `education`, `other_info` — each `null` if
absent.

With ~1200 rows, generating one at a time means ~1200 separate forward
passes through the model — most of that time is the GPU sitting
underused between calls (tokenizing, parsing, Python overhead), not
actually computing. **Batched generation** processes `EXTRACT_BATCH_SIZE`
rows in a single `model.generate()` call instead, which keeps the GPU
doing real work continuously rather than idling between small calls.

Two things batching needs that single-row generation didn't:
- **Left padding** (set on the tokenizer above) — every row in a batch is
  padded to the same length before the batch's real content, so
  generation continues from one shared position for every row at once.
- **Per-row retry for parse failures** — if one row in a batch produces
  malformed JSON, the whole batch doesn't get discarded; only that
  specific row gets re-generated individually afterward, using the same
  single-row `generate_json` helper as a fallback.

In [5]:
# ---- Shared helper: one chat completion + JSON parsing (used as a
# fallback for rows that fail batched parsing) ----
def _extract_json_block(text: str) -> str:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise json.JSONDecodeError("no JSON object found in output", text, 0)
    return match.group(0)


def generate_json(system_prompt: str, user_prompt: str, max_new_tokens: int, retries: int = 2) -> dict:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    raw_output, last_error = "", None
    for attempt in range(retries + 1):
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=DO_SAMPLE,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        try:
            return json.loads(_extract_json_block(raw_output))
        except json.JSONDecodeError as e:
            last_error = e
            if attempt < retries:
                continue

    return {"_parse_error": f"PARSE_ERROR: {last_error}", "_raw_output": raw_output}


# ---- Tracks how many tokens each row\'s generation actually used, before
# hitting max_new_tokens -- this is what the length diagnostic after
# extraction reads from, to check whether the ceiling is set sensibly. ----
GENERATION_TOKEN_LENGTHS = []


# ---- Batched version: generates for a whole list of prompts in one
# model.generate() call ----
def generate_json_batch(system_prompt: str, user_prompts: list, max_new_tokens: int) -> list:
    chat_texts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": up}],
            tokenize=False, add_generation_prompt=True,
        )
        for up in user_prompts
    ]

    inputs = tokenizer(chat_texts, return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Left padding means every row's real input ends at the same position,
    # so generated tokens for every row in the batch start at this shared
    # offset.
    input_len = inputs["input_ids"].shape[1]

    results = []
    for row_ids in output_ids:
        new_tokens = row_ids[input_len:]
        # Real length = tokens generated before the first pad/eos token --
        # this is how many steps this row actually needed, not the ceiling.
        real_length = int((new_tokens != tokenizer.pad_token_id).sum())
        GENERATION_TOKEN_LENGTHS.append(real_length)

        raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        try:
            results.append(json.loads(_extract_json_block(raw_output)))
        except json.JSONDecodeError:
            results.append({"_parse_error": "PARSE_ERROR", "_raw_output": raw_output})
    return results

In [6]:
EXTRACTION_EXAMPLES = """
Input: "ms in ai gen ai prompt engineering machine learning deep learning nlp hugging face transformers llms langchain rag asr model development python sql tableau azure certified interspeech publication"
{"job_title": "machine learning engineer", "experience": "student/intern-level", "education": "ms in ai", "other_info": "gen ai, prompt engineering, machine learning, deep learning, nlp, hugging face, transformers, llms, langchain, rag, asr, model development, python, sql, tableau, azure certified, interspeech publication"}

Input: "recent graduate in data science from University of Waterloo"
{"job_title": "data scientist", "experience": "student/intern-level", "education": "university of waterloo", "other_info": null}

Input: "software test engineer at amdocs artificial intelligence machine learning i deep learning neural network llm gen ai"
{"job_title": "software test engineer", "experience": null, "education": null, "other_info": "ai/ml/deep learning/neural networks/llm/genai skills, works at amdocs"}

Input: "senior product manager at microsoft agile scrum stakeholder management product strategy roadmapping"
{"job_title": "senior product manager", "experience": "senior-level", "education": null, "other_info": "agile, scrum, stakeholder management, product strategy skills, works at microsoft"}

Input: "data analyst at deloitte sql python tableau power bi excel dashboards reporting"
{"job_title": "data analyst", "experience": "student/intern-level", "education": null, "other_info": "sql, python, tableau, power bi skills, works at deloitte"}

Input: "HR Manager with 5 years experience"
{"job_title": "human resources manager", "experience": "5 years", "education": null, "other_info": null}

Input: "Senior Data Scientist | PhD Statistics, Stanford"
{"job_title": "senior data scientist", "experience": "senior-level", "education": "phd statistics, stanford", "other_info": null}

Input: "Passionate about helping people and solving problems every day"
{"job_title": null, "experience": null, "education": null, "other_info": null}

Input: "michigan state university alumni open for data scienceanalytics roles skilled in python r machine learning"
{"job_title": "data scientist", "experience": "student/intern-level", "education": "michigan state university (alumni)", "other_info": "python, r, machine learning skills"}

Input: "data science python sql tableau data storytelling"
{"job_title": "data scientist", "experience": "student/intern-level", "education": null, "other_info": "python, sql, tableau, data storytelling skills"}

Input: "Experienced marketing coordinator with a passion for brand storytelling, 3+ years"
{"job_title": "marketing coordinator", "experience": "3+ years", "education": null, "other_info": null}

Input: "Student at XYZ University studying finance, seeking internship opportunities"
{"job_title": "finance", "experience": "student/intern-level", "education": "xyz university", "other_info": null}

Input: "ms in data analytics engineering northeastern university boston"
{"job_title": "data analyst", "experience": "student/intern-level", "education": "ms in data analytics engineering, northeastern university, boston", "other_info": null}
""".strip()

EXTRACTION_SYSTEM_PROMPT = """You extract structured information from a
candidate's headline, bio sentence, or job title field.

Extract exactly these four fields:
- "job_title": the role phrase itself. NORMALIZE common abbreviations so
  equivalent roles produce the same string ("HR" -> "human resources",
  "PM" -> "project manager", "SWE" -> "software engineer",
  "BA" -> "business analyst", "QA" -> "quality assurance"). If no role is
  explicitly stated but the text describes a student or graduate studying
  a specific field (e.g. "studying finance", "CS student", "data science
  major"), infer the job title they are most likely pursuing from that
  field of study (e.g. "financial analyst", "software engineer", "data
  scientist") rather than using the field name itself as the title. Only
  do this when a field of study is actually named — do not guess a title
  from vague interests or personality statements. null if no role and no
  identifiable field of study.
- "experience": a seniority or years-of-experience signal ONLY if stated
  or clearly implied directly in the text itself. Use "student/intern-level"
  when the text identifies the person as a current student, recent
  graduate with no stated work history, or explicitly seeking an
  internship. null if absent.
- "education": university, degree, or program mentioned. null if absent.
- "other_info": certifications or other clearly useful signal. null if
  absent.

All string values (job_title, experience, education, other_info) must be
entirely lowercase, regardless of how the input text is capitalized. Do
not lowercase or otherwise alter the input text itself — only the values
you extract.

Never invent information that isn't present in the text. Respond with a
JSON object in exactly this format, nothing else:
{"job_title": <string or null>, "experience": <string or null>, "education": <string or null>, "other_info": <string or null>}

Examples:
""" + EXTRACTION_EXAMPLES


def extract_candidate_info(raw_text: str) -> dict:
    result = generate_json(
        EXTRACTION_SYSTEM_PROMPT, f'Input: "{raw_text}"', max_new_tokens=MAX_NEW_TOKENS_EXTRACT,
    )
    return {
        "job_title": result.get("job_title"),
        "experience": result.get("experience"),
        "education": result.get("education"),
        "other_info": result.get("other_info"),
        "_error": result.get("_parse_error"),
    }

In [7]:
def extract_batch(raw_texts: list) -> list:
    user_prompts = [f'Input: "{t}"' for t in raw_texts]
    raw_results = generate_json_batch(EXTRACTION_SYSTEM_PROMPT, user_prompts, MAX_NEW_TOKENS_EXTRACT)

    for i, r in enumerate(raw_results):
        if "_parse_error" in r:
            raw_results[i] = generate_json(EXTRACTION_SYSTEM_PROMPT, user_prompts[i], MAX_NEW_TOKENS_EXTRACT)

    return [
        {
            "job_title": r.get("job_title"),
            "experience": r.get("experience"),
            "education": r.get("education"),
            "other_info": r.get("other_info"),
            "_error": r.get("_parse_error"),
        }
        for r in raw_results
    ]




In [8]:
# ---- Run extraction across the full cleaned dataset, in batches ----
# NOTE: screening_score / location are carried straight through from df --
# these are needed as fine-tuning labels/inputs later and are NOT
# something extraction produces, so they must be preserved here rather
# than re-derived downstream. There is no target_job_title column in this
# dataset (raw_texts_all is the candidate\'s own title) -- job_title itself
# is the only title signal, extracted per-candidate below.
raw_texts_all = df["title"].tolist()
ids_all = df["id"].tolist()
screening_scores_all = df["screening_score"].tolist()
locations_all = df["location"].tolist() if "location" in df.columns else [None] * len(df)

extracted_records = []
for start in range(0, len(raw_texts_all), EXTRACT_BATCH_SIZE):
    chunk_texts = raw_texts_all[start:start + EXTRACT_BATCH_SIZE]
    chunk_ids = ids_all[start:start + EXTRACT_BATCH_SIZE]
    chunk_scores = screening_scores_all[start:start + EXTRACT_BATCH_SIZE]
    chunk_locations = locations_all[start:start + EXTRACT_BATCH_SIZE]
    chunk_results = extract_batch(chunk_texts)

    for id_, text, score, loc, res in zip(
        chunk_ids, chunk_texts, chunk_scores, chunk_locations, chunk_results
    ):
        extracted_records.append({
            "id": id_,
            "raw_text": text,
            "screening_score": score,
            "location": loc,
            **res,
        })

    done = min(start + EXTRACT_BATCH_SIZE, len(raw_texts_all))
    print(f"Extracted {done}/{len(raw_texts_all)} rows...")

print("Rows with parse errors:", sum(1 for r in extracted_records if r["_error"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Extracted 25/1266 rows...
Extracted 50/1266 rows...
Extracted 75/1266 rows...
Extracted 100/1266 rows...
Extracted 125/1266 rows...
Extracted 150/1266 rows...
Extracted 175/1266 rows...
Extracted 200/1266 rows...
Extracted 225/1266 rows...
Extracted 250/1266 rows...
Extracted 275/1266 rows...
Extracted 300/1266 rows...
Extracted 325/1266 rows...
Extracted 350/1266 rows...
Extracted 375/1266 rows...
Extracted 400/1266 rows...
Extracted 425/1266 rows...
Extracted 450/1266 rows...
Extracted 475/1266 rows...
Extracted 500/1266 rows...
Extracted 525/1266 rows...
Extracted 550/1266 rows...
Extracted 575/1266 rows...
Extracted 600/1266 rows...
Extracted 625/1266 rows...
Extracted 650/1266 rows...
Extracted 675/1266 rows...
Extracted 700/1266 rows...
Extracted 725/1266 rows...
Extracted 750/1266 rows...
Extracted 775/1266 rows...
Extracted 800/1266 rows...
Extracted 825/1266 rows...
Extracted 850/1266 rows...
Extracted 875/1266 rows...
Extracted 900/1266 rows...
Extracted 925/1266 rows...
Extr

### Blank `experience` / `education` / `other_info` — sample rows

Same idea as above, for the other three fields. A high blank rate here
isn't necessarily wrong (many `raw_text` values genuinely don't mention
experience or education) — the point of the sample is to eyeball whether
that's actually true for these rows, or whether the prompt is missing
information that's plainly present in the text.

In [9]:
# ---- Save the extracted JSON dataset ----
with open(EXTRACTED_OUTPUT_PATH, "w") as f:
    json.dump(extracted_records, f, indent=2)

extracted_df = pd.DataFrame(extracted_records)
print(f"Saved {EXTRACTED_OUTPUT_PATH} ({len(extracted_df)} records)")
extracted_df.head()

Saved /kaggle/working/candidate_extracted.json (1266 records)


,id,raw_text,screening_score,location,job_title,experience,education,other_info,_error
0,1,innovative and driven professional seeking a r...,100,United States,data scientist,student/intern-level,None,None,None
1,2,ms applied data science student usc research a...,100,United States,data scientist,student/intern-level,"ms applied data science, usc","research assistant, usc, former data analytics...",None
2,3,computer science student seeking full-time sof...,100,United States,software engineer,student/intern-level,computer science student,"ai, sql, data visualization, python, ssrs",None
3,4,microsoft certified power bi data analyst mba ...,100,United States,data analyst,student/intern-level,"mba, unt business intelligence engineering, ms...","power bi, business analytics, predictive analy...",None
4,5,graduate research assistant at uab masters in ...,100,United States,data scientist,student/intern-level,"masters in data science, uab",None,None


In [10]:
from IPython.display import display

FIELDS_TO_CHECK = ["job_title", "experience", "education", "other_info"]


def is_blank(value) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    if isinstance(value, str) and not value.strip():
        return True
    return False


print(f"Total extracted rows: {len(extracted_df)}\n")
print("Missing/blank counts per field:")
for field in FIELDS_TO_CHECK:
    n_blank = extracted_df[field].apply(is_blank).sum()
    pct = 100 * n_blank / len(extracted_df)
    print(f"  {field:12s}: {n_blank:4d} blank  ({pct:5.1f}%)")

n_parse_errors = extracted_df["_error"].apply(lambda v: not is_blank(v)).sum()
print(f"\nRows with a JSON parse error (_error set): {n_parse_errors}")

n_fully_blank = extracted_df[FIELDS_TO_CHECK].apply(
    lambda row: all(is_blank(v) for v in row), axis=1
).sum()
print(f"Rows where ALL four fields are blank: {n_fully_blank}")

Total extracted rows: 1266

Missing/blank counts per field:
  job_title   :   75 blank  (  5.9%)
  experience  :  508 blank  ( 40.1%)
  education   :  712 blank  ( 56.2%)
  other_info  :  624 blank  ( 49.3%)

Rows with a JSON parse error (_error set): 0
Rows where ALL four fields are blank: 26


In [13]:
import numpy as np

lengths = np.array(GENERATION_TOKEN_LENGTHS)
print(f"Rows measured: {len(lengths)}")
print(f"Mean tokens generated:  {lengths.mean():.1f}")
print(f"Median tokens generated: {np.median(lengths):.1f}")
print(f"95th percentile:        {np.percentile(lengths, 95):.1f}")
print(f"Max tokens generated:   {lengths.max()}")
print(f"Current MAX_NEW_TOKENS_EXTRACT: {MAX_NEW_TOKENS_EXTRACT}")

if np.percentile(lengths, 95) < MAX_NEW_TOKENS_EXTRACT * 0.7:
    print(f"\nMAX_NEW_TOKENS_EXTRACT looks higher than needed -- consider "
          f"lowering it to around {int(np.percentile(lengths, 95)) + 5} "
          f"(95th percentile + a small buffer) on the next run.")

Rows measured: 1266
Mean tokens generated:  62.5
Median tokens generated: 61.0
95th percentile:        84.0
Max tokens generated:   87
Current MAX_NEW_TOKENS_EXTRACT: 90


In [14]:
parse_error_df = extracted_df[extracted_df["_error"].apply(lambda v: not is_blank(v))]
pct_errors = 100 * len(parse_error_df) / len(extracted_df)
print(f"{len(parse_error_df)} rows with a parse error ({pct_errors:.1f}% of data)\n")

pd.set_option("display.max_colwidth", 120)
display(parse_error_df[["id", "raw_text", "_error"]].head(30))

0 rows with a parse error (0.0% of data)



,id,raw_text,_error


In [15]:
PARSE_ERROR_REVIEW_PATH = "/kaggle/working/parse_error_rows.csv"
parse_error_df.to_csv(PARSE_ERROR_REVIEW_PATH, index=False)
print(f"Saved {len(parse_error_df)} parse-error rows to {PARSE_ERROR_REVIEW_PATH} for review")

Saved 0 parse-error rows to /kaggle/working/parse_error_rows.csv for review
